In [2]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys

In [4]:
#Entry
tperiod = 8 #time period is 4 hours long

In [6]:
# Location data
df_V = pd.read_excel('fullDist Set V Data Location.xlsx')
df_V.head

<bound method NDFrame.head of     Index      County       Type  Cluster  \
0       0         NaN  Warehouse      NaN   
1       1         NaN  Warehouse      4.0   
2       2         NaN   Supplier      NaN   
3       3         NaN   Supplier      NaN   
4       4         NaN   Supplier      NaN   
5       5         NaN   Supplier      NaN   
6       6  Georgetown    Shelter      9.0   
7       7  Georgetown    Shelter      1.0   
8       8       Horry    Shelter     18.0   
9       9       Horry    Shelter      2.0   
10     10       Horry    Shelter     16.0   
11     11       Horry    Shelter      8.0   
12     12    Berkeley    Shelter     12.0   
13     13    Berkeley    Shelter     17.0   
14     14    Berkeley    Shelter     14.0   
15     15    Berkeley    Shelter     11.0   
16     16    Berkeley    Shelter     19.0   
17     17  Charleston    Shelter      0.0   
18     18  Charleston    Shelter     15.0   
19     19  Dorchester    Shelter      7.0   
20     20  Dorchester    

In [8]:
# County data
df_C = pd.read_excel('fullDist Set C Data.xlsx')
df_C.head

<bound method NDFrame.head of        County Category of Hurricane  Shelter Total Demand  PoD Total Demand  \
0    Berkeley                 Cat 2                  6400             11596   
1  Charleston                 Cat 2                  5103              9246   
2    Colleton                 Cat 1                  1650              2500   
3  Dorchester                 Cat 1                  4460              8080   
4  Georgetown                 Cat 3                  2970              5368   
5       Horry                 Cat 3                 20234             36574   
6      Jasper                   NaN                     0                 0   

   % Evac to PubShelter  Unnamed: 5  Unnamed: 6  Unnamed: 7 All cots:  \
0                 0.190         NaN         NaN         NaN    Water    
1                 0.190         NaN         NaN         NaN     Meals   
2                 0.109         NaN         NaN         NaN       Cot   
3                 0.190         NaN         N

In [10]:
# Resource Data
df_R = pd.read_excel('fullDist R.xlsx')
df_R.head

<bound method NDFrame.head of              R  Demand Units on Truck  Pallet Units on Truck  \
0  Shelter Kit                    768                     44   
1      PoD Kit                   1728                     34   
2        Water                   2912                     26   
3        Meals                   4608                     48   
4          Cot                   3120                     52   
5      Blanket                   2496                     52   

   pallet per S-Kit  pallet per P-Kit  kit pallet meeting demand  
0          0.000000          0.000000                   0.057292  
1          0.000000          0.000000                   0.019676  
2          0.008929          0.008929                   0.000000  
3          0.010417          0.010417                   0.000000  
4          0.020833          0.000000                   0.000000  
5          0.016667          0.000000                   0.000000  >

In [12]:
#Time and distance matrices 
timeMatrix = np.loadtxt("durationFULL.txt"); # load the time duration matrix (seconds)
distanceMatrix = np.loadtxt("distanceFULL.txt");

In [14]:
df_inputs = pd.read_excel('fullDistInputs.xlsx')
df_inputs.head

<bound method NDFrame.head of    Hours in 6 days  S Curve k Value  People on Truck  Hourly Salary  \
0            144.0              0.2              4.0           15.0   
1              NaN              NaN              NaN            NaN   
2              NaN              NaN              NaN            NaN   
3              NaN              NaN              NaN            NaN   
4              NaN              NaN              NaN            NaN   
5              NaN              NaN              NaN            NaN   

   Fuel Cost Per Mile  Open Cost V0  Open Cost V1  Open Cost V2  Open Cost V3  \
0                 0.1           0.0        1000.0        1250.0         500.0   
1                 NaN           NaN           NaN           NaN           NaN   
2                 NaN           NaN           NaN           NaN           NaN   
3                 NaN           NaN           NaN           NaN           NaN   
4                 NaN           NaN           NaN           NaN    

In [16]:
# Load the K_ijt data from the Excel file
df_K = pd.read_excel('K_ijt_data_fixed.xlsx')
df_K.head()

,From_Node,To_Node,Time_Period,Capacity
0,0,0,0,98
1,0,0,1,58
2,0,0,2,101
3,0,0,3,186
4,0,0,4,132


In [18]:
# Estimated demand
file_path = 'demand_data_.xlsx'
sheet_names = ['Demand_1', 'Demand_2','Demand_3','Demand_4','Demand_5']

d_its = {}

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(file_path, sheet_name=sheet, header=0, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    demand_array = df.values

    for i in range(demand_array.shape[0]):
        actual_location = i  # Start location index from 1 instead of 0
        for t in range(demand_array.shape[1]):
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 4:", d_its[(6, 0, 4)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(42, 17, 1)])


Sample demand data for location 6, time 1, scenario 4: 66.53350405727443
Sample demand data for location 6, time 1, scenario 2: 0.0


In [20]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 8
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Define scenario indices for 50 scenarios
sce = list(range(5))  # This will create a list [0, 1, 2, ..., 49]

# Define equal probabilities for each of the 50 scenarios
P = [1/5] * 5  # This will create a list with 50 elements, each with a value of 1/50

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 7
All locations: 43
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 21 [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
RSA locations: 9 [27, 28, 29, 30, 31, 32, 33, 34, 35]
PoD locations: 7 [36, 37, 38, 39, 40, 41, 42]
All resources: 6
Time periods: 18
Time Before Landfall: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Time After Landfall: [9, 10, 11, 12, 13, 14, 15, 16, 17]
Scenario indices: [0, 1, 2, 3, 4]
Scenario probabilities: [0.2, 0.2, 0.2, 0.2, 0.2]


In [22]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[23, 42]
Number of arcs: 1400
Number of arcs eliminated by this arc set: 449


In [24]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])

#starting inventory I_ir0[I][R]
Inv = []
for i in range(V):
    Inv.extend([df_V['S-Kit StartInv'][i],
                df_V['P-Kit StartInv'][i],
                df_V['Water StartInv'][i],
                df_V['Food StartInv'][i],
                df_V['Cot StartInv'][i],
                df_V['Blanket StartInv'][i]])
I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)

#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])

#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print(K_ijt)
#print (Inv)
print(g_shelter)
print(g_pod)

[0.0, 0.0, 0.008928571428571428, 0.010416666666666666, 0.020833333333333332, 0.016666666666666666]
[0.0, 0.0, 0.008928571428571428, 0.010416666666666666, 0.0, 0.0]


# Defining Master Problem

In [27]:
# Create the master problem model
master_model = gp.Model("MasterProblem")

# Define the inventory decision variables I[i, r, t] for first-stage decisions
# These represent the inventory of commodity r at location i at the start of time period t
I = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = master_model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"I_{i}_{r}_{t}")

# Define the auxiliary variable θ with an initial lower bound, used for Benders cuts
theta = master_model.addVar(vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="theta")

# Set an initial lower bound for theta to prevent unboundedness from below
L = -1e5  # Choose a reasonable initial lower bound based on your problem
master_model.addConstr(theta >= L, name="theta_lower_bound")

# Set the objective function to minimize θ (representing the expected second-stage cost)
master_model.setObjective(theta, GRB.MINIMIZE)

# Set output flag to 0 to avoid printing Gurobi logs
master_model.setParam("OutputFlag", 0);

# Update the model to integrate new variables
master_model.update()

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


# Defining Optimality Subproblem

In [30]:
# Define the optimality subproblem
sub_opt = gp.Model("subprob_optimality")
sub_opt.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the optimality subproblem
U_opt = {}
S_opt = {}
f_opt = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            U_opt[i, t, s] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"U_opt_{i}_{t}_{s}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            S_opt[i, t, s] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"S_opt_{i}_{t}_{s}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    for s in range(len(sce)):
                        f_opt[i, j, r, t, s] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"f_opt_{i}_{j}_{r}_{t}_{s}")

# Objective: Minimize the expected demand shortage cost
#sub_opt.setObjective(gp.quicksum(P[s] * gp.quicksum(p_i[i] * U_opt[i, t, s] for i in range(V) for t in range(T)) for s in sce))

# Constraint for inventory flow balance with U and S
pi_opt_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Start from t=1 since t=0 is the initial condition
        for s in range(len(sce)):
            pi_opt_inventory_balance_U[i, t, s] = sub_opt.addConstr(
                U_opt[i, t, s] - U_opt[i, t-1, s] + S_opt[i, t, s] == d_its[(i, t, s)],
                name=f"inventory_balance_U_{i}_{t}_{s}"
            )

# Initial shortage for demand at t=0
pi_opt_initial_shortage_U = {}
for i in range(V):
    for s in range(len(sce)):
        pi_opt_initial_shortage_U[i, s] = sub_opt.addConstr(
            U_opt[i, 0, s] == d_its[(i, 0, s)],
            name=f"initial_shortage_U_{i}_{s}"
        )

# Demand satisfaction constraint
pi_opt_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            pi_opt_demand_satisfaction[i, t, s] = sub_opt.addConstr(
                U_opt[i, t, s] - S_opt[i, t, s] >= 0,
                name=f"demand_satisfaction_{i}_{t}_{s}"
            )

# Ensure that V is an iterable range of locations
V_range = range(V)

# Flow conservation constraint for f with the RHS set to 0 temporarily, using appropriate g_shelter or g_pod
pi_opt_flow_conservation = {}
for i in V_range:
    for r in range(R):
        for t in range(T - 1):
            for s in range(len(sce)):
                # Determine which g value to use based on the type of location
                g_value = g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0  # Assuming g_value should be zero for locations not in V1 or V3

                pi_opt_flow_conservation[i, r, t, s] = sub_opt.addConstr(
                    -gp.quicksum(f_opt[i, j, r, t, s] for j in V_range if (i, j) in A)
                    + gp.quicksum(f_opt[j, i, r, t - t_ijt[j][i] + 1, s] for j in V_range if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                    - S_opt[i, t, s] * g_value == 0,
                    name=f"flow_conservation_{i}_{r}_{t}_{s}"
                )

# Inventory balance constraint with RHS set to 0 for now
pi_opt_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(sce)):
                # Determine which g value to use based on the type of location
                g_value = g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0  # Assuming g_value should be zero for locations not in V1 or V3
                
                # Add the constraint to the model
                pi_opt_inventory_balance[i, r, t, s] = sub_opt.addConstr(
                    -gp.quicksum(f_opt[j, i, r, t, s] for j in V_range if (j, i) in A)
                    - S_opt[i, t, s] * g_value >= 0,
                    name=f"inventory_balance_{i}_{r}_{t}_{s}"
                )


# Arc capacity constraint
pi_opt_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in range(len(sce)):
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        pi_opt_arc_capacity[i, j, t, s] = sub_opt.addConstr(
                            -gp.quicksum(w_r[r] * f_opt[i, j, r, t, s] for r in range(R)) >= -K_ijt[i][j][t],
                            name=f"arc_capacity_{i}_{j}_{t}_{s}"
                        )

# Defining Feasibility Subproblem

In [33]:
# Define the feasibility subproblem
sub_feas = gp.Model("subprob_feasibility")
sub_feas.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the feasibility subproblem
U_feas = {}
S_feas = {}
f_feas = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            U_feas[i, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"U_feas_{i}_{t}_{s}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            S_feas[i, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"S_feas_{i}_{t}_{s}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    for s in range(len(sce)):
                        f_feas[i, j, r, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"f_feas_{i}_{j}_{r}_{t}_{s}")

# Now define slack variables ε (epsilon) for each group in the objective function
# ε variables with objective coefficients corresponding to their respective constraint group
epsilon_1 = {}
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            epsilon_1[i, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon1_{i}_{t}_{s}")

epsilon_2 = {}
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            epsilon_2[i, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon2_{i}_{t}_{s}")

epsilon_3 = {}
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            epsilon_3[i, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon3_{i}_{t}_{s}")

epsilon_4 = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(sce)):
                epsilon_4[i, r, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon4_{i}_{r}_{t}_{s}")

epsilon_5 = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(sce)):
                epsilon_5[i, r, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon5_{i}_{r}_{t}_{s}")

epsilon_6 = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(sce)):
                epsilon_6[i, r, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon6_{i}_{r}_{t}_{s}")

epsilon_7 = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for t in range(T):
                for s in range(len(sce)):
                    epsilon_7[i, j, t, s] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=1, name=f"epsilon7_{i}_{j}_{t}_{s}")


# Constraint: Inventory flow balance for U, S, and epsilon variables
pi_feas_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Note: starting from t=1 as per the constraint
        for s in range(len(sce)):
            pi_feas_inventory_balance_U[i, t, s] = sub_feas.addConstr(
                U_feas[i, t, s] - U_feas[i, t-1, s] + S_feas[i, t, s] + epsilon_1[i, t, s] - epsilon_2[i, t, s] == d_its[(i, t, s)],
                name=f"inventory_balance_U_feas_{i}_{t}_{s}"
            )
            
# Demand satisfaction constraint
pi_feas_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        for s in range(len(sce)):
            pi_feas_demand_satisfaction[i, t, s] = sub_feas.addConstr(
                U_feas[i, t, s] - S_feas[i, t, s] + epsilon_3[i, t, s] >= 0,
                name=f"demand_satisfaction_{i}_{t}_{s}"
            )

# Ensure that V is an iterable range of locations
V_range = range(V)

# Flow conservation constraint
pi_feas_flow_conservation = {}
for i in V_range:
    for r in range(R):
        for t in range(T - 1):
            for s in range(len(sce)):
                pi_feas_flow_conservation[i, r, t, s] = sub_feas.addConstr(
                    -gp.quicksum(f_feas[i, j, r, t, s] for j in V_range if (i, j) in A)
                    + gp.quicksum(f_feas[j, i, r, t - t_ijt[j][i] + 1, s] for j in V_range if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                    - S_feas[i, t, s] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                    + epsilon_4[i, r, t, s] - epsilon_5[i, r, t, s] == 0,
                    name=f"flow_conservation_{i}_{r}_{t}_{s}"
                )

# Constraint for flow and demand satisfaction with epsilon_6
pi_feas_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in range(len(sce)):
                g_value = g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0  # Determine g based on the location type
                pi_feas_inventory_balance[i, r, t, s] = sub_feas.addConstr(
                    -gp.quicksum(f_feas[i, j, r, t, s] for j in V_range if (i, j) in A)
                    - S_feas[i, t, s] * g_value
                    + epsilon_6[i, r, t, s] >= 0,  # Right-hand side set to 0 for now
                    name=f"inventory_balance_{i}_{r}_{t}_{s}"
                )

# Flow capacity constraint for all links, time periods, and scenarios
pi_feas_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in range(len(sce)):
                    # Check if K_ijt[i][j][t] exists before adding the constraint
                    if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        pi_feas_arc_capacity[i, j, t, s] = sub_feas.addConstr(
                            -gp.quicksum(w_r[r] * f_feas[i, j, r, t, s] for r in range(R))
                            + epsilon_7[i, j, t, s] >= -K_ijt[i][j][t],
                            name=f"arc_capacity_{i}_{j}_{t}_{s}"
                        )

# Creating Benders Loop

In [38]:
LB = -1e10
UB = 1e10

Ocuts = 0
Fcuts = 0

iter = 0
e = 1e-5  # tolerance parameter

while (UB - LB) / UB > e:
    iter += 1
    
    # First, solve the master problem to get the values of first-stage variables and theta
    I_values = {}  # Store I values for each (i, r, t)
    theta_value = 0  # Initialize theta value

    # Optimize the master problem
    master_model.optimize()

    # Check if the optimization was successful
    if master_model.status == GRB.OPTIMAL:
        # Extract the values of first-stage decision variables I[i, r, t]
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    I_values[i, r, t] = I[i, r, t].x  # Get the optimized value of I[i, r, t]

        # Get the value of theta
        theta_value = theta.x  # Retrieve the optimized value of theta

        # Update the lower bound
        LB = master_model.objVal  # Update the lower bound with the objective value of the master problem
    
    elif master_model.status == GRB.INFEASIBLE:
        print("Master problem is infeasible.")
        break  # Or handle infeasibility as required for your problem

    elif master_model.status == GRB.UNBOUNDED:
        print("Master problem is unbounded.")
        break  # Or handle unboundedness as required for your problem

    else:
        print(f"Optimization was stopped with status {master_model.status}")
        break

    # Number of scenarios
    num = len(sce)

    # Initialize zero dual multipliers for each constraint type in the optimality subproblem as dictionaries
    pi_opt_inventory_balance_U_val = {}
    pi_opt_initial_shortage_U_val = {}
    pi_opt_demand_satisfaction_val = {}
    pi_opt_flow_conservation_val = {}
    pi_opt_inventory_balance_val = {}
    pi_opt_arc_capacity_val = {}

    # Inventory balance U constraint duals (only store for (i, t, s) where it applies)
    for i in range(V):
        for t in range(1, T):  # Skip t=0 as it might be handled separately
            for s in range(num):
                pi_opt_inventory_balance_U_val[(i, t, s)] = 0.0
    
    # Initial shortage U constraint duals
    for i in range(V):
        for s in range(num):
            pi_opt_initial_shortage_U_val[(i, s)] = 0.0
    
    # Demand satisfaction duals
    for i in range(V):
        for t in range(T):
            for s in range(num):
                pi_opt_demand_satisfaction_val[(i, t, s)] = 0.0
    
    # Flow conservation constraint duals
    for i in range(V):
        for r in range(R):
            for t in range(T - 1):
                for s in range(num):
                    pi_opt_flow_conservation_val[(i, r, t, s)] = 0.0
    
    # Inventory balance constraint duals
    for i in range(V):
        for r in range(R):
            for t in range(T):
                for s in range(num):
                    pi_opt_inventory_balance_val[(i, r, t, s)] = 0.0
    
    # Arc capacity constraint duals - already handled as a dictionary over valid arcs
    for i, j in A:
        for t in range(T):
            for s in range(num):
                pi_opt_arc_capacity_val[(i, j, t, s)] = 0.0  # Initialize dual multiplier to zero

    # Initialize zero dual multipliers for each constraint type in the feasibility subproblem as dictionaries
    
    # Inventory balance U constraint duals for feasibility (only store for relevant (i, t, s) indices)
    pi_feas_inventory_balance_U_val = {}
    for i in range(V):
        for t in range(1, T):  # Skip t=0 as it might be handled separately
            for s in range(num):
                pi_feas_inventory_balance_U_val[(i, t, s)] = 0.0
    
    # Demand satisfaction constraint duals for feasibility
    pi_feas_demand_satisfaction_val = {}
    for i in range(V):
        for t in range(T):
            for s in range(num):
                pi_feas_demand_satisfaction_val[(i, t, s)] = 0.0
    
    # Flow conservation constraint duals for feasibility
    pi_feas_flow_conservation_val = {}
    for i in range(V):
        for r in range(R):
            for t in range(T - 1):
                for s in range(num):
                    pi_feas_flow_conservation_val[(i, r, t, s)] = 0.0
    
    # Inventory balance constraint duals for feasibility
    pi_feas_inventory_balance_val = {}
    for i in range(V):
        for r in range(R):
            for t in range(T):
                for s in range(num):
                    pi_feas_inventory_balance_val[(i, r, t, s)] = 0.0
    
    # Arc capacity constraint duals for feasibility - store only for valid arcs in A
    pi_feas_arc_capacity_val = {}
    for i, j in A:
        for t in range(T):
            for s in range(num):
                pi_feas_arc_capacity_val[(i, j, t, s)] = 0.0  # Initialize dual multiplier to zero

    # Initialize a vector to store the optimal objective value for the optimality subproblem for each scenario
    Q_opt = np.zeros(num)
    # Initialize a vector to store the optimal objective value for the feasibility subproblem for each scenario
    w_feas = np.zeros(num)
    FLAG = 0; #(0 if I_hat is feasible ∀ s, and 1 otherwise)

    
    # Update the RHS of the flow conservation constraint in the optimality subproblem
    for i in V_range:
        for r in range(R):
            for t in range(T - 1):  # Loop up to T-1 since we need I[t+1] for the RHS
                for s in range(len(sce)):
                    # Calculate the RHS value as I_values[i, r, t+1] - I_values[i, r, t]
                    if (i, r, t+1) in I_values and (i, r, t) in I_values:
                        pi_opt_flow_conservation[i, r, t, s].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])

    # Update the RHS of the inventory balance constraint in the optimality subproblem
    for i in range(V):
        for r in range(R):
            for t in range(T):
                for s in range(len(sce)):
                    if (i, r, t) in I_values:
                        pi_opt_inventory_balance[i, r, t, s].setAttr(GRB.Attr.RHS, -I_values[i, r, t])

    for s in range(num):
        sub_opt.optimize()
    
        if sub_opt.status == GRB.OPTIMAL:
            # Update the optimal objective value for each scenario
            Q_opt[s] = sub_opt.objVal
    
            # Retrieve and store dual values for each relevant constraint
            for key, constraint in pi_opt_inventory_balance_U.items():
                pi_opt_inventory_balance_U_val[key] = constraint.pi
            for key, constraint in pi_opt_initial_shortage_U.items():
                pi_opt_initial_shortage_U_val[key] = constraint.pi
            for key, constraint in pi_opt_demand_satisfaction.items():
                pi_opt_demand_satisfaction_val[key] = constraint.pi
            for key, constraint in pi_opt_flow_conservation.items():
                pi_opt_flow_conservation_val[key] = constraint.pi
            for key, constraint in pi_opt_inventory_balance.items():
                pi_opt_inventory_balance_val[key] = constraint.pi
            for key, constraint in pi_opt_arc_capacity.items():
                pi_opt_arc_capacity_val[key] = constraint.pi
    
        elif sub_opt.status == GRB.INFEASIBLE or sub_opt.status == GRB.INF_OR_UNBD:
            # Set FLAG to 1 if infeasible, indicating the need for a feasibility cut
            FLAG = 1
            break;
        else:
            # Handle other statuses
            print('One of the scenario subproblems is neither optimal nor infeasible')
            print('status =', sub_opt.status)
            exit(0)

    if FLAG == 1:
        # Initialize the dictionary to store RHS values for the feasibility constraints
        rhs_feas = {}
        
        # Update the RHS of the flow conservation constraints
        for i in V_range:
            for r in range(R):
                for t in range(T - 1):  # Loop up to T-1 as we need I[t+1] for the RHS
                    for s in range(len(sce)):
                        # Calculate the RHS value as I_values[i, r, t+1] - I_values[i, r, t] for flow conservation
                        if (i, r, t+1) in I_values and (i, r, t) in I_values:
                            pi_feas_flow_conservation[i, r, t, s].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
        
        # Update the RHS of the inventory balance constraint
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    for s in range(len(sce)):
                        if (i, r, t) in I_values:
                            pi_feas_inventory_balance[i, r, t, s].setAttr(GRB.Attr.RHS, -I_values[i, r, t])

        for s in range(num):
            # Optimize the feasibility subproblem
            sub_feas.optimize()
            w_feas[s] = sub_feas.objVal  # Store the optimal objective value for this scenario

            #Populating rhs of the constraints
            # Inventory flow balance for U and S constraints
            for i in range(V):
                for t in range(1, T):
                    for s in range(len(sce)):
                        rhs_feas[f"inventory_balance_U_feas_{i}_{t}_{s}"] = d_its[(i, t, s)]
            
            # Demand satisfaction constraint
            for i in range(V):
                for t in range(T):
                    for s in range(len(sce)):
                        rhs_feas[f"demand_satisfaction_{i}_{t}_{s}"] = 0
            
            # Flow conservation constraint
            for i in V_range:
                for r in range(R):
                    for t in range(T - 1):
                        for s in range(len(sce)):
                            if (i, r, t+1) in I_values and (i, r, t) in I_values:
                                rhs_feas[f"flow_conservation_{i}_{r}_{t}_{s}"] = I_values[i, r, t+1] - I_values[i, r, t]
            
            # Inventory balance constraint
            for i in range(V):
                for r in range(R):
                    for t in range(T):
                        for s in range(len(sce)):
                            if (i, r, t) in I_values:
                                rhs_feas[f"inventory_balance_{i}_{r}_{t}_{s}"] = -I_values[i, r, t]
            
            # Flow capacity constraint for all links, time periods, and scenarios
            for i in range(V):
                for j in range(V):
                    if (i, j) in A:
                        for t in range(T):
                            for s in range(len(sce)):
                                if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                                    rhs_feas[f"arc_capacity_{i}_{j}_{t}_{s}"] = -K_ijt[i][j][t]


            # Retrieve and store dual values for each relevant constraint in the feasibility subproblem
            for key, constraint in pi_feas_inventory_balance_U.items():
                pi_feas_inventory_balance_U_val[key] = constraint.pi            
            for key, constraint in pi_feas_demand_satisfaction.items():
                pi_feas_demand_satisfaction_val[key] = constraint.pi            
            for key, constraint in pi_feas_flow_conservation.items():
                pi_feas_flow_conservation_val[key] = constraint.pi            
            for key, constraint in pi_feas_inventory_balance.items():
                pi_feas_inventory_balance_val[key] = constraint.pi            
            for key, constraint in pi_feas_arc_capacity.items():
                pi_feas_arc_capacity_val[key] = constraint.pi

            if w_feas[s] > e:
                
                #Add the feasibility cut
                feasibility_cut_expr = gp.LinExpr()
                for key, dual_val in pi_feas_inventory_balance_U_val.items():
                    if key in rhs_feas:
                        feasibility_cut_expr += dual_val * rhs_feas[key]
                for key, dual_val in pi_feas_demand_satisfaction_val.items():
                    if key in rhs_feas:
                        feasibility_cut_expr += dual_val * rhs_feas[key]        
                for key, dual_val in pi_feas_flow_conservation_val.items():
                    if key in rhs_feas:
                        feasibility_cut_expr += dual_val * rhs_feas[key]
                for key, dual_val in pi_feas_inventory_balance_val.items():
                    if key in rhs_feas:
                        feasibility_cut_expr += dual_val * rhs_feas[key]
                for key, dual_val in pi_feas_arc_capacity_val.items():
                    if key in rhs_feas:
                        feasibility_cut_expr += dual_val * rhs_feas[key]
        
                # Add the feasibility cut to the master problem (left-hand side of the cut is zero)
                master_model.addConstr(feasibility_cut_expr <= 0, name=f"feasibility_cut_scenario_{s}")
                Fcuts += 1  # Increment the counter for feasibility cuts
                
                break

    else:
        # Calculate the temporary upper bound using only the second-stage costs
        tempUB = sum(Q_opt[s] * P[s] for s in range(num))        
        if tempUB < UB:
            UB = tempUB
        if (UB - LB) / max(1e-10, abs(LB)) < e:
            break

        else:
            # Initialize the dictionary to store RHS values for the optimality constraints
            rhs_opt = {}
            # Populate RHS for inventory flow balance with U and S constraints
            for i in range(V):
                for t in range(1, T):
                    for s in range(num):
                        rhs_opt[f"inventory_balance_U_{i}_{t}_{s}"] = d_its[(i, t, s)]  # Based on demand
            
            # Initial shortage for demand at t=0
            for i in range(V):
                for s in range(num):
                    rhs_opt[f"initial_shortage_U_{i}_{s}"] = d_its[(i, 0, s)]
            
            # Demand satisfaction constraint
            for i in range(V):
                for t in range(T):
                    for s in range(num):
                        rhs_opt[f"demand_satisfaction_{i}_{t}_{s}"] = 0  # RHS is 0 for this constraint
            
            # Flow conservation constraint
            for i in V_range:
                for r in range(R):
                    for t in range(T - 1):
                        for s in range(num):
                            if (i, r, t + 1) in I_values and (i, r, t) in I_values:
                                rhs_opt[f"flow_conservation_{i}_{r}_{t}_{s}"] = I_values[i, r, t + 1] - I_values[i, r, t]
            
            # Inventory balance constraint
            for i in range(V):
                for r in range(R):
                    for t in range(T):
                        for s in range(num):
                            if (i, r, t) in I_values:
                                rhs_opt[f"inventory_balance_{i}_{r}_{t}_{s}"] = -I_values[i, r, t]
            
            # Flow capacity constraint for all links, time periods, and scenarios
            for i in range(V):
                for j in range(V):
                    if (i, j) in A:  # Only consider valid links in A
                        for t in range(T):
                            for s in range(num):
                                if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                                    rhs_opt[f"arc_capacity_{i}_{j}_{t}_{s}"] = -K_ijt[i][j][t]
                                    
            # Add the optimality cut
            for s in range(num):
                optimality_cut_expr = gp.LinExpr()
                
                # Multiply dual values with the corresponding rhs_opt values for each constraint
                for key, dual_val in pi_opt_inventory_balance_U_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]                
                for key, dual_val in pi_opt_initial_shortage_U_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]                
                for key, dual_val in pi_opt_demand_satisfaction_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]                
                for key, dual_val in pi_opt_flow_conservation_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]                
                for key, dual_val in pi_opt_inventory_balance_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]                
                for key, dual_val in pi_opt_arc_capacity_val.items():
                    if key in rhs_opt:
                        optimality_cut_expr += dual_val * rhs_opt[key]
                
                # Multiply by scenario probabilities and add the cut to the master model
                master_model.addConstr(theta >= optimality_cut_expr * P[s], name=f"optimality_cut_scenario_{s}")
                Ocuts += 1  # Increment the counter for optimality cuts

    # Print information at each iteration
    print("iter =", iter)
    print("FLAG =", FLAG)
    print("LB =", LB)
    print("UB =", UB)
    print("===============================================================")

# Final summary after convergence
print("#########################################################################################")
print("#########################################################################################")
print("Final iteration =", iter)
print("Number of feasibility cuts added =", Fcuts)
print("Number of optimality cuts added =", Ocuts)
print("Lower Bound (LB) =", LB)
print("Upper Bound (UB) =", UB)

# Print first-stage decision variable values (I_values)
print("I_values (First-stage decision variables):")
for key, value in I_values.items():
    print(f"I[{key}] =", value)

Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 57660 rows, 763740 columns and 2248690 nonzeros
Model fingerprint: 0xd059df13
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e+00, 1e+04]
Presolve removed 57660 rows and 763740 columns
Presolve time: 0.30s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      1s

Solved in 0 iterations and 0.50 seconds (0.31 work units)
Optimal objective  0.000000000e+00
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thr

C:\Users\hkhakan\AppData\Local\Temp\ipykernel_22576\1678868023.py:10: RuntimeWarning: divide by zero encountered in scalar divide
  while (UB - LB) / UB > e:


Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 57660 rows, 763740 columns and 2248690 nonzeros
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e+00, 1e+04]
LP warm-start: use basis
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.29 seconds (0.05 work units)
Optimal objective  0.000000000e+00
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 57660 rows, 7